# Stacking a star and validating the PSF products

The empirical test of the PSF products: take an *isolated* star (found with
`scripts/gaia_isolated_stars.py`), stack all its cutouts on a common grid,
and compare the stack against the pipeline PSF.

**Prerequisite** — download an isolated star (this one comes from the
Gaia search near (180, +30)):

```bash
python scripts/download_cutouts.py --ra 179.455173 --dec 29.362296 --size 23
```

In [ ]:
import sys
sys.path.append('..')

import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from reproject import reproject_adaptive
from matplotlib import pyplot as plt

from src.ztf_data import load_query, local_paths

In [ ]:
ra, dec = 179.455173, 29.362296   # isolated G~12 star (gaia_isolated_stars.py)
size = 23                         # cutout half-size (arcsec)
max_seeing = 1.7

zquery = load_query(ra, dec, size, max_seeing)
paths = local_paths(zquery)
print(f'{len(paths["sciimg.fits"])} cutouts on disk')

## 1. Resample every cutout onto a common grid

Each cutout has its own WCS; the star sits at a slightly different sub-pixel
position in each. We build one target WCS with (ra, dec) at the exact center
pixel and reproject every cutout onto it (flux-conserving adaptive
resampling), so the star is centered identically everywhere.

In [ ]:
PIXSCALE = 1.012 / 3600  # ZTF pixel scale (deg/px)

def make_target_wcs(ra, dec, size, pixscale=PIXSCALE):
    """Tangent-plane WCS with (ra, dec) at the exact center pixel."""
    w = WCS(naxis=2)
    w.wcs.crpix = [(size + 1) / 2, (size + 1) / 2]
    w.wcs.crval = [ra, dec]
    w.wcs.cdelt = [-pixscale, pixscale]
    w.wcs.ctype = ['RA---TAN', 'DEC--TAN']
    return w

target_wcs = make_target_wcs(ra, dec, size)

stars, seeing = [], []
for f in paths['sciimg.fits']:
    hdu = fits.open(f)[0]
    star, _ = reproject_adaptive(
        (hdu.data, WCS(hdu.header)),
        target_wcs, shape_out=(size, size),
        kernel='hann', boundary_mode='constant', conserve_flux=True,
    )
    star = np.nan_to_num(star, nan=0.0)
    star -= star.min()
    star /= star.sum()
    stars.append(star)
    seeing.append(hdu.header['SEEING'])

stars = np.array(stars)
seeing = np.array(seeing)
print(f'{len(stars)} cutouts resampled; '
      f'seeing {seeing.min():.2f}-{seeing.max():.2f}" '
      f'(median {np.median(seeing):.2f}")')

## 2. Stack

With every exposure normalized to unit flux and centered, the mean is an
empirical PSF (averaged over the seeing distribution of the survey).

In [ ]:
stack = stars.mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
im0 = axes[0].imshow(stack, origin='lower')
axes[0].set_title(f'mean of {len(stars)} exposures')
plt.colorbar(im0, ax=axes[0], shrink=0.8)
im1 = axes[1].imshow(np.log10(np.maximum(stack, 1e-6)), origin='lower')
axes[1].set_title('log10 stretch')
plt.colorbar(im1, ax=axes[1], shrink=0.8)
plt.tight_layout()
plt.show()

## 3. Compare against the pipeline PSF

Pick the exposures closest to the median seeing and compare their pipeline
center-PSF (cropped to the cutout size) with the empirical stack of the same
exposures. Residuals at the few-percent level are expected: the stack
averages over sub-pixel resampling and background noise, and the pipeline
PSF is defined at the quadrant center rather than at the star's position
(see notebook 01 for the position-dependent correction).

In [ ]:
def crop_center(img, out):
    c = img.shape[0] // 2
    h = out // 2
    return img[c - h:c + h + 1, c - h:c + h + 1]

med = np.median(seeing)
sel = np.abs(seeing - med) < 0.05
print(f'{sel.sum()} exposures within 0.05" of median seeing {med:.2f}"')

stack_sel = stars[sel].mean(axis=0)
psfs = []
for f in np.array(paths['sciimgdaopsfcent.fits'])[sel]:
    p = fits.getdata(f).astype(float)
    p = crop_center(p, size)
    psfs.append(p / p.sum())
psf_sel = np.mean(psfs, axis=0)

resid = stack_sel - psf_sel
rmax = np.abs(resid).max()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, title, kw in [
    (axes[0], stack_sel, 'empirical stack', {}),
    (axes[1], psf_sel, 'pipeline center PSF', {}),
    (axes[2], resid, f'residual (max={rmax:.4f})',
     dict(cmap='RdBu_r', vmin=-rmax, vmax=rmax)),
]:
    im = ax.imshow(img, origin='lower', **kw)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()